# **Air University Multan Campus**<br>

**LAB 5 Task<br>**

**Subject Name & Code:** SE425 Data Mining<br>
**Department:** Computer Science<br>
**Program:** BS Software Engineering<br>
**Semester:** 7 Fall 2026<br>
**Instructor:** Fazle Hanan<br>
**Date & Week:** 7 October 2026 & Week 5<br>
**Lab:** B216<br>

### Student Details

Save this notebook as `RegID_Name_Lab5Task.ipynb` before you submit it.

**Registration ID:** 233670

**Name:** Sidra Ghulam Hussain


Complete the 3 tasks below. Write your code under each `# TODO` line and run the cell. No expected output is given — read each task carefully and check your own results. Do not load any file other than the provided `headlines_task5.csv` and `penguins.csv`. Marking criteria are in the Lab 5 Task Handout rubric.

In [2]:
# Setup cell — run this first. Do not modify.
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

headlines_df = pd.read_csv("headlines_task5.csv")
penguins_df = pd.read_csv("penguins.csv")
print(headlines_df.shape, penguins_df.shape)
headlines_df.head()

(6, 2) (342, 7)


,headline,topic
0,city opens new health clinic for local families,health
1,doctors urge families to get flu vaccine early,health
2,airline adds new flights to the northern coast,travel
3,tourists book more flights as coastal hotels r...,travel
4,university adds new course on data science,education


In [3]:
# Import SciPy, a library that provides scientific and statistical functions
import scipy

# Import scikit-learn, a library of data mining and machine learning tools
import sklearn

# Import Counter to count how many times each word appears
from collections import Counter

# Import combinations to loop over every unique pair of objects
from itertools import combinations

# Import the text-vectorising and cosine functions from scikit-learn
# CountVectorizer  → turns headlines into a term-count matrix
# cosine_similarity → calculates cosine similarity between rows of a matrix
from sklearn.feature_extraction.text import CountVectorizer

# Import the cosine distance function from SciPy
# (cosine distance = 1 - cosine similarity)
from scipy.spatial.distance import cosine

### Task 1.1 — Building the term-count matrix (10 marks)

Using the `headline` column of `headlines_df`:
1. Split every headline into lowercase words (split on spaces; do not remove any word).
2. Build a sorted list of the distinct words and store it as `vocab`.
3. Build the term-count matrix as a NumPy array `C` with one row per headline and one column per word in `vocab`, without using scikit-learn.
4. Also store it as a DataFrame called `C_df` with the rows labelled `t1` to `t6` and the columns named by `vocab`.
5. Print the shape of `C` and the number of words in `vocab`.

In [4]:
# TODO: build vocab, the NumPy array C (6 headlines x vocabulary) and the DataFrame C_df (rows t1..t6)
# Split each headline into individual lowercase words
# Example: "City Opens New Clinic" → ["city", "opens", "new", "clinic"]
word_lists = [text.lower().split() for text in headlines_df["headline"]]

# Collect all different words and arrange them alphabetically
vocab = sorted({word for words in word_lists for word in words})

# Give each headline a simple name: t1, t2, ..., t6
names = [f"t{i+1}" for i in range(len(headlines_df))]

# Count how many times each word appears in each headline (no scikit-learn)
# Rows = headlines (t1, ..., t6), Columns = vocabulary words
C = np.array([[words.count(word) for word in vocab]
              for words in word_lists])

# Convert the matrix into a labelled table
C_df = pd.DataFrame(C, index=names, columns=vocab)

# Print the size of the matrix and the number of unique words
print("Term-count matrix shape:", C.shape)
print("Vocabulary size:", len(vocab))

# Display the full term-count matrix
C_df

Term-count matrix shape: (6, 35)
Vocabulary size: 35


,adds,airline,as,book,city,clinic,coast,coastal,course,data,...,reopen,science,students,the,to,tourists,university,urge,vaccine,welcome
t1,0,0,0,0,1,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
t2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,1,1,0
t3,1,1,0,0,0,0,1,0,0,0,...,0,0,0,1,1,0,0,0,0,0
t4,0,0,1,1,0,0,0,1,0,0,...,1,0,0,0,0,1,0,0,0,0
t5,1,0,0,0,0,0,0,0,1,1,...,0,1,0,0,0,0,1,0,0,0
t6,0,0,0,0,0,0,0,0,1,1,...,0,1,1,0,0,0,0,0,0,1


In [5]:
# Create the term-count matrix using scikit-learn
# CountVectorizer does two things automatically:
# 1. Finds the vocabulary (unique words)
# 2. Counts each word in each headline
vectorizer = CountVectorizer()

# Convert the headlines into a numerical matrix
# Rows = headlines, Columns = words
C_sklearn = vectorizer.fit_transform(headlines_df["headline"]).toarray()

# Check whether CountVectorizer found the same vocabulary as our first method
# True = both methods found exactly the same words in the same order
print("Same vocabulary:", list(vectorizer.get_feature_names_out()) == vocab)

# Check whether both methods produced exactly the same word counts
# True = both matrices contain the same values
print("Same counts:", np.array_equal(C, C_sklearn))

Same vocabulary: True
Same counts: True


### Task 2.1 — Cosine similarity: manual vs scikit-learn (40 marks)

1. Write a function `manual_cosine(a, b)` that computes the cosine similarity between two 1-D NumPy arrays directly from the formula (no `scikit-learn`, no `scipy`).
2. For every pair of the 6 headlines in `C`, print the manual result next to the result from `sklearn.metrics.pairwise.cosine_similarity` for the same pair (`cosine_similarity` expects 2-D input).
3. Print the largest absolute difference between the manual and scikit-learn results across all pairs.
4. Print the most similar pair of headlines and its cosine similarity.

In [6]:
# TODO: define manual_cosine(a, b), compare it against sklearn's cosine_similarity for every pair in C, and report the most similar pair

# Import combinations to loop over every unique pair of headlines
from itertools import combinations


# Calculate cosine similarity between two headline vectors
def manual_cosine(a, b):

    # Dot product: multiply matching word counts and add them
    dot_product = np.sum(a * b)

    # Length (magnitude) of each vector
    length_a = np.sqrt(np.sum(a ** 2))
    length_b = np.sqrt(np.sum(b ** 2))

    # Cosine similarity = dot product / (length of a × length of b)
    return dot_product / (length_a * length_b)


# Store the largest difference between manual and scikit-learn results
max_diff = 0.0

# Store every pair with its manual similarity to find the best pair later
pair_results = []

# Print the column headings
print(f"{'Pair':8}{'manual':>10}{'sklearn':>10}")

# Compare every possible pair of different headlines
for i, j in combinations(range(len(C)), 2):

    # Cosine similarity using our own formula
    manual = manual_cosine(C[i], C[j])

    # Cosine similarity using scikit-learn
    # reshape changes the vector into the required 2-D format
    library_sklearn = cosine_similarity(
        C[i].reshape(1, -1), C[j].reshape(1, -1)
    )[0, 0]

    # Keep the largest difference between the two methods
    max_diff = max(max_diff, abs(manual - library_sklearn))

    # Save the pair and its similarity
    pair_results.append((names[i], names[j], manual))

    # Print the manual result next to the scikit-learn result
    print(f"{names[i]}-{names[j]:5}{manual:10.4f}{library_sklearn:10.4f}")

# Show the largest difference found across ALL pairs
# A value close to 0 means the methods agree
print(f"\nLargest |manual - sklearn| difference: {max_diff:.2e}")

# Find the most similar pair (highest cosine similarity)
best_a, best_b, best_value = max(pair_results, key=lambda p: p[2])
print(f"Most similar pair: {best_a}-{best_b} (cosine similarity {best_value:.4f})")

Pair        manual   sklearn
t1-t2       0.1250    0.1250
t1-t3       0.1250    0.1250
t1-t4       0.0000    0.0000
t1-t5       0.1336    0.1336
t1-t6       0.1443    0.1443
t2-t3       0.1250    0.1250
t2-t4       0.0000    0.0000
t2-t5       0.0000    0.0000
t2-t6       0.0000    0.0000
t3-t4       0.1250    0.1250
t3-t5       0.2673    0.2673
t3-t6       0.1443    0.1443
t4-t5       0.0000    0.0000
t4-t6       0.0000    0.0000
t5-t6       0.6172    0.6172

Largest |manual - sklearn| difference: 0.00e+00
Most similar pair: t5-t6 (cosine similarity 0.6172)


In [8]:
# Calculate cosine similarity between every pair of headlines
# The result is an 6 × 6 matrix
# Each row and column represents a headline (t1 to t6)
sim_matrix = cosine_similarity(C)

# Convert the result into a labelled table
# Round the similarity values to 3 decimal places
sim_df = pd.DataFrame(
    np.round(sim_matrix, 3),
    index=names,
    columns=names
)

# Display the cosine similarity matrix
print("Cosine similarity matrix:")
sim_df

Cosine similarity matrix:


,t1,t2,t3,t4,t5,t6
t1,1.000,0.125,0.125,0.000,0.134,0.144
t2,0.125,1.000,0.125,0.000,0.000,0.000
t3,0.125,0.125,1.000,0.125,0.267,0.144
t4,0.000,0.000,0.125,1.000,0.000,0.000
t5,0.134,0.000,0.267,0.000,1.000,0.617
t6,0.144,0.000,0.144,0.000,0.617,1.000


In [9]:
# Find the most similar pair(s) of headlines
def most_similar_pairs(matrix, labels):

    # Create all unique headline pairs with their similarity
    # Example: (h1, h2, similarity)
    pairs = [(labels[i], labels[j], matrix[i, j])
             for i, j in combinations(range(len(labels)), 2)]

    # Find the highest similarity value
    best = max(p[2] for p in pairs)

    # Return all pairs having the highest similarity
    # np.isclose handles very small rounding differences
    return [p for p in pairs if np.isclose(p[2], best)]


# Find the most similar headline pair(s)
best_pairs = most_similar_pairs(sim_matrix, names)

# Display the most similar pair(s)
for a, b, s in best_pairs:
    print(f"Most similar pair: {a}-{b} (cosine similarity {s:.4f})")


# Get the similarity value for every unique pair
# There are 8 × 7 / 2 = 28 unique pairs
pair_values = [sim_matrix[i, j]
               for i, j in combinations(range(len(names)), 2)]

# Count pairs with similarity 0
# Similarity 0 means the two headlines share no words
zero_pairs = sum(1 for v in pair_values if np.isclose(v, 0))

print(f"Pairs with similarity 0: {zero_pairs} of {len(pair_values)}")

Most similar pair: t5-t6 (cosine similarity 0.6172)
Pairs with similarity 0: 6 of 15


### Task 3.1 — Mixed-type dissimilarity matrix (50 marks)

Using `penguins_df`:
1. Drop rows with missing values in `bill_length_mm` or `body_mass_g`, then reset the index.
2. Take the penguins at index positions 10, 40, 70, 240 and 300 (positions after the drop and reset), keep only the columns `species`, `island`, `bill_length_mm` and `body_mass_g`, and store the result as a DataFrame called `S_df` with the rows labelled `s1` to `s5`.
3. Build the 5×5 mixed-type dissimilarity matrix. For each pair of objects, compute one value per attribute: for the nominal attributes (`species`, `island`) it is 0 if the two values are the same and 1 if they differ; for the numeric attributes it is |x − y| divided by (max − min) of that attribute, with max and min taken over the 5 objects in `S_df`. The dissimilarity of the pair is the average of the 4 per-attribute values.
4. Print the matrix as a labelled pandas DataFrame (rows and columns labelled `s1`–`s5`), rounded to 3 decimal places.
5. Print which pair of objects is closest and which pair is furthest apart.

In [ ]:
# TODO: build S_df and the 5x5 mixed-type dissimilarity matrix, print it rounded to 3 decimals, and report the closest/furthest pairs

# Remove rows where bill length or body mass is missing
# reset_index() creates new row positions after removing missing rows
clean = penguins_df.dropna(
    subset=["bill_length_mm", "body_mass_g"]
).reset_index(drop=True)

# Select 5 penguins using their row positions
# Keep the 4 attributes needed for mixed-type dissimilarity
S_df = clean.iloc[
    [10, 40, 70, 240, 300]
][
    ["species", "island", "bill_length_mm", "body_mass_g"]
].copy()

# Rename the 5 selected penguins as s1, s2, ..., s5
S_df.index = [f"s{i+1}" for i in range(5)]

# Display the mixed-type data matrix
print("Mixed-type data matrix (5 objects × 4 attributes):")
print(S_df)

# Separate the attributes into nominal and numeric groups
nominal_cols = ["species", "island"]
numeric_cols = ["bill_length_mm", "body_mass_g"]

# Range (max - min) of each numeric attribute, taken over the 5 selected penguins
ranges = {
    col: float(S_df[col].max() - S_df[col].min())
    for col in numeric_cols
}
print("Ranges:", ranges)


# Calculate the dissimilarity of each attribute for two objects
def attribute_dissimilarities(row_a, row_b):

    # Store the dissimilarity value for each of the 4 attributes
    d = []

    # Nominal attributes: same value → 0, different value → 1
    for col in nominal_cols:
        d.append(0.0 if row_a[col] == row_b[col] else 1.0)

    # Numeric attributes: d = |x1 - x2| / (max - min)
    for col in numeric_cols:
        d.append(abs(row_a[col] - row_b[col]) / ranges[col])

    # Return the 4 attribute dissimilarities
    return d


# Calculate one combined dissimilarity by averaging the 4 attribute values
def mixed_dissimilarity(row_a, row_b):
    d = attribute_dissimilarities(row_a, row_b)
    return sum(d) / len(d)


# Number of penguins (objects)
m = len(S_df)

# Create an empty 5 × 5 matrix
mixed_matrix = np.zeros((m, m))

# Calculate dissimilarity between every pair of penguins
for i in range(m):
    for j in range(m):
        mixed_matrix[i, j] = mixed_dissimilarity(S_df.iloc[i], S_df.iloc[j])

# Convert the matrix into a labelled table, rounded to 3 decimal places
mixed_df = pd.DataFrame(
    np.round(mixed_matrix, 3),
    index=S_df.index,
    columns=S_df.index
)

# Display the mixed-type dissimilarity matrix
print("\nMixed-type dissimilarity matrix:")
print(mixed_df)


# Find the closest and furthest pair(s) from the dissimilarity matrix
def closest_and_furthest(matrix, labels):

    # All unique pairs with their dissimilarity value
    pairs = [
        (labels[i], labels[j], matrix[i, j])
        for i, j in combinations(range(len(labels)), 2)
    ]

    # Smallest and largest dissimilarity values
    smallest = min(p[2] for p in pairs)
    largest = max(p[2] for p in pairs)

    # np.isclose() handles very small floating-point differences
    closest = [p for p in pairs if np.isclose(p[2], smallest)]
    furthest = [p for p in pairs if np.isclose(p[2], largest)]

    return closest, furthest


# Find the closest and furthest penguin pair(s)
closest, furthest = closest_and_furthest(mixed_matrix, list(S_df.index))

# Smaller dissimilarity → more similar objects
print("Closest pair:", ", ".join(f"{a}-{b}" for a, b, _ in closest),
      f"({closest[0][2]:.3f})")

# Larger dissimilarity → more different objects
print("Furthest pair:", ", ".join(f"{a}-{b}" for a, b, _ in furthest),
      f"({furthest[0][2]:.3f})")

Full dataset shape: (342, 7)

Mixed-type data matrix (5 objects × 4 attributes):


,species,island,bill_length_mm,body_mass_g
s1,Adelie,Torgersen,37.8,3700.0
s2,Adelie,Dream,40.8,3900.0
s3,Adelie,Torgersen,39.7,3900.0
s4,Gentoo,Biscoe,45.1,5000.0
s5,Gentoo,Biscoe,52.5,5450.0
